# Peterson-McCabe: Sentence Influence in Free Personal Narratives

**Question:** Does long-range influence structure develop with age in **free personal narratives** — where there is no external scaffolding (no picture book, no predefined story)?

**Why this matters:** The ECSC analysis (frog stories) showed no developmental effect on long-range hotspot density — likely because the shared picture book provides external scaffolding that masks individual differences in memory capacity. Personal narratives depend entirely on the child's own memory to maintain coherence.

**Data:** Peterson & McCabe corpus — 1092 personal narratives from children ages 4-9, elicited through conversation. Children tell stories about things that happened to them.

**Prediction:** Stronger age effect than ECSC. Older children should show:
- More long-range hotspots (they maintain more information across their narrative)
- Longer sentence lifespans
- Higher mean influence at distances 5+

**Note:** Texts are shorter (mean ~84 words, ~6 sentences) so maximum distances are limited. We use a lower word threshold and analyze all available narratives.

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/petmcc_processed")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/PetMcc_influence")
    if (DRIVE_DATA / "transcripts.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/petmcc_processed")
        if not (LOCAL_DATA / "transcripts.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            print("Upload transcripts.jsonl:")
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/petmcc_influence")
    DATA_DIR = Path("../data/petmcc_processed")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

MIN_WORDS = 50      # lower threshold for shorter personal narratives
MIN_SENTS = 5       # need at least 5 sentences for meaningful influence
MAX_SENTENCES = 20   # cap per doc
N_DOCS_PER_BIN = 50  # more docs since they're shorter/faster
RANDOM_SEED = 42

AGE_BINS = [
    (48, 66, '4-5y'),
    (67, 84, '5-7y'),
    (85, 113, '7-9y'),
]

print(f"Min words: {MIN_WORDS}, Min sents: {MIN_SENTS}")
print(f"Docs per age bin: {N_DOCS_PER_BIN}")
print(f"Age bins: {[b[2] for b in AGE_BINS]}")

In [ ]:
corpus_all = []
with open(DATA_DIR / "transcripts.jsonl") as f:
    for line in f:
        d = json.loads(line)
        pop = json.loads(d['population'])
        d['age_months'] = pop.get('age_months', None)
        d['sex'] = pop.get('sex', None)
        d['word_count'] = len(d['text'].split())
        sents = re.split(r'(?<=[.!?])\s+', d['text'].strip())
        d['n_sents'] = len([s for s in sents if len(s.strip().split()) >= 3])
        corpus_all.append(d)

# Filter
corpus_all = [d for d in corpus_all
              if d['age_months'] is not None
              and d['word_count'] >= MIN_WORDS
              and d['n_sents'] >= MIN_SENTS]
print(f"After filtering: {len(corpus_all)} transcripts")

# Assign age bins
for d in corpus_all:
    d['age_bin'] = None
    for lo, hi, label in AGE_BINS:
        if lo <= d['age_months'] <= hi:
            d['age_bin'] = label
            break
corpus_all = [d for d in corpus_all if d['age_bin'] is not None]

# Sample N per age bin
rng = np.random.RandomState(RANDOM_SEED)
corpus = []
for lo, hi, label in AGE_BINS:
    pool = [d for d in corpus_all if d['age_bin'] == label]
    n = min(len(pool), N_DOCS_PER_BIN)
    chosen = list(rng.choice(pool, size=n, replace=False))
    corpus.extend(chosen)

print(f"Selected {len(corpus)} transcripts")
for lo, hi, label in AGE_BINS:
    sub = [d for d in corpus if d['age_bin'] == label]
    ages = [d['age_months'] for d in sub]
    wcs = [d['word_count'] for d in sub]
    sents = [d['n_sents'] for d in sub]
    print(f"  {label}: n={len(sub)}, age={min(ages)}-{max(ages)}mo, "
          f"words={np.mean(wcs):.0f} mean, sents={np.mean(sents):.1f} mean")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

In [ ]:
def split_into_sentences(text):
    # More lenient for child speech — accept shorter sentences
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip().split()) >= 3]


@torch.no_grad()
def compute_ppl_on_target(context_token_ids, target_token_ids):
    full_ids = context_token_ids + target_token_ids
    input_ids = torch.tensor([full_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    target_start = len(context_token_ids)
    total_loss = 0.0
    count = 0
    for i in range(target_start, len(full_ids) - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[full_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def compute_influence_matrix(doc):
    sentences = split_into_sentences(doc['text'])
    if len(sentences) < 4:
        return None, None
    if len(sentences) > MAX_SENTENCES:
        sentences = sentences[:MAX_SENTENCES]

    n = len(sentences)
    sent_ids = [tokenizer.encode(s, add_special_tokens=False) for s in sentences]
    matrix = np.full((n, n), np.nan)

    for t in range(2, n):
        target_ids = sent_ids[t]
        if len(target_ids) < 2:
            continue
        full_context_ids = []
        for k in range(t):
            full_context_ids.extend(sent_ids[k])
        ppl_full = compute_ppl_on_target(full_context_ids, target_ids)
        if math.isinf(ppl_full) or ppl_full <= 0:
            continue
        for i in range(t):
            dropped_context_ids = []
            for k in range(t):
                if k != i:
                    dropped_context_ids.extend(sent_ids[k])
            if len(dropped_context_ids) < 2:
                continue
            ppl_dropped = compute_ppl_on_target(dropped_context_ids, target_ids)
            if math.isinf(ppl_dropped) or ppl_dropped <= 0:
                continue
            matrix[i, t] = math.log(ppl_dropped / ppl_full)

    return sentences, matrix


print("Functions defined")

In [ ]:
results_path = BASE_DIR / "petmcc_influence_matrices.npz"
meta_path = BASE_DIR / "petmcc_influence_meta.json"

if results_path.exists() and meta_path.exists():
    loaded = np.load(results_path, allow_pickle=True)
    all_matrices = list(loaded['matrices'])
    with open(meta_path) as f:
        all_meta = json.load(f)
    print(f"Loaded {len(all_matrices)} influence matrices")
else:
    all_matrices = []
    all_meta = []
    for doc in tqdm(corpus, desc="Computing influence matrices"):
        sentences, matrix = compute_influence_matrix(doc)
        if matrix is None:
            continue
        all_matrices.append(matrix)
        all_meta.append({
            'doc_id': doc['doc_id'],
            'age_months': doc['age_months'],
            'age_bin': doc['age_bin'],
            'n_sentences': len(sentences),
            'word_count': doc['word_count'],
            'sentences': sentences,
        })

    np.savez(results_path, matrices=np.array(all_matrices, dtype=object))
    with open(meta_path, 'w') as f:
        json.dump(all_meta, f)
    print(f"Computed {len(all_matrices)} matrices, saved to {BASE_DIR}")

print(f"Documents: {len(all_matrices)}")
for _, _, label in AGE_BINS:
    sub = [m for m in all_meta if m['age_bin'] == label]
    n_sents = [m['n_sentences'] for m in sub]
    print(f"  {label}: {len(sub)} docs, {np.mean(n_sents):.1f} mean sents")

In [ ]:
# Aggregate by distance and age
by_dist_age = {label: {} for _, _, label in AGE_BINS}

for doc_idx, matrix in enumerate(all_matrices):
    n = matrix.shape[0]
    age_bin = all_meta[doc_idx]['age_bin']
    for i in range(n):
        for t in range(i+1, n):
            val = matrix[i, t]
            if not np.isnan(val):
                d = t - i
                if d not in by_dist_age[age_bin]:
                    by_dist_age[age_bin][d] = []
                by_dist_age[age_bin][d].append(val)

print("INFLUENCE BY DISTANCE AND AGE:")
print(f"{'Dist':>5}", end='')
for _, _, label in AGE_BINS:
    print(f"  {label+' mean':>10} {label+' %+':>7} {label+' n':>6}", end='')
print()
print("-" * 80)
for d in range(1, 18):
    has_data = False
    for _, _, label in AGE_BINS:
        if len(by_dist_age[label].get(d, [])) >= 3:
            has_data = True
    if not has_data:
        continue
    print(f"{d:>5}", end='')
    for _, _, label in AGE_BINS:
        vals = np.array(by_dist_age[label].get(d, []))
        if len(vals) >= 3:
            print(f"  {vals.mean():>10.4f} {100*np.mean(vals>0):>6.1f}% {len(vals):>6}", end='')
        else:
            print(f"  {'':>10} {'':>7} {'':>6}", end='')
    print()

print()
print("HOTSPOT DENSITY BY AGE:")
for thresh in [0.05, 0.1, 0.2]:
    for min_d in [3, 5]:
        print(f"  influence>{thresh}, dist>={min_d}:")
        for _, _, label in AGE_BINS:
            count = total = 0
            for d in by_dist_age[label]:
                if d >= min_d:
                    vals = np.array(by_dist_age[label][d])
                    count += np.sum(vals > thresh)
                    total += len(vals)
            pct = 100*count/total if total > 0 else 0
            print(f"    {label}: {count}/{total} = {pct:.1f}%")

print()
print("LIFESPAN BY AGE:")
for _, _, label in AGE_BINS:
    lifespans = []
    for doc_idx, matrix in enumerate(all_matrices):
        if all_meta[doc_idx]['age_bin'] != label:
            continue
        n = matrix.shape[0]
        for i in range(n-2):
            last_d = 0
            for t in range(i+1, n):
                if not np.isnan(matrix[i,t]) and matrix[i,t] > 0.01:
                    last_d = t - i
            if last_d > 0:
                lifespans.append(last_d)
    lifespans = np.array(lifespans) if lifespans else np.array([0])
    print(f"  {label}: mean={lifespans.mean():.1f}, >3: {100*np.mean(lifespans>3):.1f}%, >5: {100*np.mean(lifespans>5):.1f}%")

print()
print("AGE CORRELATION (continuous):")
for d in [1, 2, 3, 5, 7, 10]:
    doc_vals = []
    doc_ages = []
    for doc_idx, matrix in enumerate(all_matrices):
        n = matrix.shape[0]
        infs = []
        for i in range(n):
            t = i + d
            if t < n and not np.isnan(matrix[i,t]):
                infs.append(matrix[i,t])
        if len(infs) >= 2:
            doc_vals.append(np.mean(infs))
            doc_ages.append(all_meta[doc_idx]['age_months'])
    if len(doc_vals) >= 10:
        r, p = stats.pearsonr(doc_ages, doc_vals)
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else ''
        print(f"  dist={d:>2}: r={r:+.3f}, p={p:.4f} {sig} (n={len(doc_vals)})")

# Overall long-range influence vs age
doc_lr = []
doc_ages_lr = []
for doc_idx, matrix in enumerate(all_matrices):
    n = matrix.shape[0]
    lr = []
    for i in range(n):
        for t in range(i+3, n):
            if not np.isnan(matrix[i,t]):
                lr.append(matrix[i,t])
    if len(lr) >= 2:
        doc_lr.append(np.mean(lr))
        doc_ages_lr.append(all_meta[doc_idx]['age_months'])
if len(doc_lr) >= 10:
    r, p = stats.pearsonr(doc_ages_lr, doc_lr)
    print(f"  Overall (dist>=3): r={r:+.3f}, p={p:.4f}")

In [ ]:
age_colors = {'4-5y': '#e74c3c', '5-7y': '#f39c12', '7-9y': '#27ae60'}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Decay by age
ax = axes[0, 0]
for _, _, label in AGE_BINS:
    dists = sorted([d for d in by_dist_age[label] if len(by_dist_age[label][d]) >= 5])
    if not dists:
        continue
    means = [np.mean(by_dist_age[label][d]) for d in dists]
    sems = [np.std(by_dist_age[label][d])/np.sqrt(len(by_dist_age[label][d])) for d in dists]
    n_docs = sum(1 for m in all_meta if m['age_bin'] == label)
    ax.errorbar(dists, means, yerr=sems, fmt='o-', color=age_colors[label],
                linewidth=2, markersize=5, capsize=2, label=f'{label} (n={n_docs})')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Sentence Distance')
ax.set_ylabel('Mean Influence')
ax.set_title('A. Influence Decay by Age (Free Narratives)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# B: Hotspot density by age
ax = axes[0, 1]
threshold = 0.05
for _, _, label in AGE_BINS:
    dists = sorted([d for d in by_dist_age[label] if len(by_dist_age[label][d]) >= 5])
    if not dists:
        continue
    pcts = [100 * np.mean(np.array(by_dist_age[label][d]) > threshold) for d in dists]
    ax.plot(dists, pcts, 'o-', color=age_colors[label], linewidth=2, markersize=5, label=label)
ax.set_xlabel('Sentence Distance')
ax.set_ylabel(f'% Pairs with Influence > {threshold}')
ax.set_title('B. Hotspot Density by Age', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# C: Age continuous vs long-range influence
ax = axes[1, 0]
if doc_lr:
    for doc_idx in range(len(doc_lr)):
        age_bin = all_meta[doc_idx]['age_bin'] if doc_idx < len(all_meta) else '4-5y'
        color = age_colors.get(age_bin, '#999')
    ax.scatter(doc_ages_lr, doc_lr, alpha=0.4, s=20,
               c=[age_colors.get(all_meta[i]['age_bin'], '#999')
                  for i in range(len(all_meta)) if i < len(doc_lr)])
    r, p = stats.pearsonr(doc_ages_lr, doc_lr)
    slope, intercept, _, _, _ = stats.linregress(doc_ages_lr, doc_lr)
    x_line = np.array([min(doc_ages_lr), max(doc_ages_lr)])
    ax.plot(x_line, intercept + slope * x_line, 'k--', linewidth=2)
    ax.set_title(f'C. Long-Range Influence vs Age (r={r:.3f}, p={p:.4f})', fontweight='bold')
ax.set_xlabel('Age (months)')
ax.set_ylabel('Mean Influence (dist >= 3)')
ax.grid(True, alpha=0.2)

# D: Compare ECSC (scaffolded) vs PetMcc (free) age effect
ax = axes[1, 1]
# Just show the age bin means for this dataset
for _, _, label in AGE_BINS:
    sub_meta = [m for m in all_meta if m['age_bin'] == label]
    if not sub_meta:
        continue
    lr_vals = []
    for doc_idx, matrix in enumerate(all_matrices):
        if all_meta[doc_idx]['age_bin'] != label:
            continue
        n = matrix.shape[0]
        lr = [matrix[i,t] for i in range(n) for t in range(i+3, n) if not np.isnan(matrix[i,t])]
        if lr:
            lr_vals.append(np.mean(lr))
    if lr_vals:
        ax.bar(label, np.mean(lr_vals), yerr=np.std(lr_vals)/np.sqrt(len(lr_vals)),
               color=age_colors[label], alpha=0.7, capsize=5)
ax.set_ylabel('Mean Long-Range Influence (dist >= 3)')
ax.set_title('D. Long-Range Influence by Age Bin', fontweight='bold')
ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('Peterson-McCabe: Sentence Influence in Free Personal Narratives\n(No External Scaffolding)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_petmcc_influence.png', dpi=150, bbox_inches='tight')
plt.show()